In [3]:
import json
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load JSON data
with open('data_players/CHI_NYK/16/players_data.json') as f:
    data_all = json.load(f)

In [4]:
#obtain the frames number where the player has been detected
frames = []
for player in data_all['action_data']['players']['player_1']['frames']:
    frames.append(player['frame_number'])

In [5]:
len(frames)

553

In [7]:
data_all['action_data']['players']['player_1']['frames']

[{'frame_number': 1,
  'bbox': {'x': 627.5, 'y': 328.7, 'width': 39.1, 'height': 122.2},
  'nbr_players_frame': 10,
  'joint_angles_2D': {'right_leg': {'ankle': 108.43,
    'knee': 167.47,
    'hip': 153.43},
   'left_leg': {'ankle': 74.05, 'knee': 167.63, 'hip': 93.58}},
  'angular_velocity_2D': {'right_leg': {'ankle': 0.0, 'knee': 0.0, 'hip': 0.0},
   'left_leg': {'ankle': 0.0, 'knee': 0.0, 'hip': 0.0}},
  'angular_acceleration_2D': {'right_leg': {'ankle': 0.0,
    'knee': 0.0,
    'hip': 0.0},
   'left_leg': {'ankle': 0.0, 'knee': 0.0, 'hip': 0.0}},
  'linear_position_2D': {'pelvis': [0.0, 0.0]},
  'linear_velocity_2D': {'pelvis': [0, 0]},
  'linear_acceleration_2D': {'pelvis': [0, 0]},
  'joint_angles_3D': {'right_leg': {'ankle': 106.32,
    'knee': 159.82,
    'hip': 92.01},
   'left_leg': {'ankle': 98.18, 'knee': 161.45, 'hip': 102.9}},
  'angular_velocity_3D': {'right_leg': {'ankle': 0.0, 'knee': 0.0, 'hip': 0.0},
   'left_leg': {'ankle': 0.0, 'knee': 0.0, 'hip': 0.0}},
  'angul

In [8]:
import numpy as np

data = data_all['action_data']['players']['player_1']

# Initialize lists to store extracted features
pose_features = []

# Extract relevant data from JSON
for i, frame in enumerate(data['frames']):
    if i == 0:
        print(frame.keys())
    
    # Extract 2D joint angles for both legs
    joint_angles_2d = [
        frame['joint_angles_2D']['right_leg']['ankle'],
        frame['joint_angles_2D']['right_leg']['knee'],
        frame['joint_angles_2D']['right_leg']['hip'],
        frame['joint_angles_2D']['left_leg']['ankle'],
        frame['joint_angles_2D']['left_leg']['knee'],
        frame['joint_angles_2D']['left_leg']['hip']
    ]
    
    # Extract 2D angular velocities for both legs
    angular_velocity_2d = [
        frame['angular_velocity_2D']['right_leg']['ankle'],
        frame['angular_velocity_2D']['right_leg']['knee'],
        frame['angular_velocity_2D']['right_leg']['hip'],
        frame['angular_velocity_2D']['left_leg']['ankle'],
        frame['angular_velocity_2D']['left_leg']['knee'],
        frame['angular_velocity_2D']['left_leg']['hip']
    ]
    
    # Extract 2D linear velocity and acceleration for the pelvis, take the norm (scalar)
    linear_velocity_2d = np.linalg.norm(frame['linear_velocity_2D']['pelvis'])
    linear_acceleration_2d = np.linalg.norm(frame['linear_acceleration_2D']['pelvis'])

    # Extract 3D linear velocity and acceleration for the pelvis, take the norm (scalar)
    linear_velocity_3d = np.linalg.norm(frame['linear_velocity_3D']['pelvis'])
    linear_acceleration_3d = np.linalg.norm(frame['linear_acceleration_3D']['pelvis'])

    # Extract 3D joint angles for both legs
    joint_angles_3d = [
        frame['joint_angles_3D']['right_leg']['hip'],
        frame['joint_angles_3D']['right_leg']['knee'],
        frame['joint_angles_3D']['right_leg']['ankle'],
        frame['joint_angles_3D']['left_leg']['hip'],
        frame['joint_angles_3D']['left_leg']['knee'],
        frame['joint_angles_3D']['left_leg']['ankle']
    ]
    
    # Extract 3D angular velocities for both legs
    angular_velocity_3d = [
        frame['angular_velocity_3D']['right_leg']['hip'],
        frame['angular_velocity_3D']['right_leg']['knee'],
        frame['angular_velocity_3D']['right_leg']['ankle'],
        frame['angular_velocity_3D']['left_leg']['hip'],
        frame['angular_velocity_3D']['left_leg']['knee'],
        frame['angular_velocity_3D']['left_leg']['ankle']
    ]

    # Extract pose 3D human and flatten it
    j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

    
    # Flatten and combine all features into one vector
    feature_vector = (
        joint_angles_2d +              # 6 elements
        angular_velocity_2d +           # 6 elements
        [linear_velocity_2d] +          # 1 element (scalar norm)
        [linear_acceleration_2d] +      # 1 element (scalar norm)
        joint_angles_3d +               # 6 elements
        angular_velocity_3d +           # 6 elements
        [linear_velocity_3d] +          # 1 element (scalar norm)
        [linear_acceleration_3d] +      # 1 element (scalar norm)
        list(j3d_human)                 # 105 elements (flattened 3D human coordinates)
    )
    
    # Append the flattened feature vector to the list
    pose_features.append(feature_vector)

# Convert to NumPy array for further processing
pose_features = np.array(pose_features, dtype=np.float32)

# Now print the shape and verify
print(pose_features.shape)  # (num_frames, total_feature_size)
print(pose_features.dtype)

dict_keys(['frame_number', 'bbox', 'nbr_players_frame', 'joint_angles_2D', 'angular_velocity_2D', 'angular_acceleration_2D', 'linear_position_2D', 'linear_velocity_2D', 'linear_acceleration_2D', 'joint_angles_3D', 'angular_velocity_3D', 'angular_acceleration_3D', 'linear_position_3D', 'linear_velocity_3D', 'linear_acceleration_3D', 'j2d', 'j3d_human'])
(553, 133)
float32


In [9]:
X_train = pose_features

In [10]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PlayerDatasetSingleFile(Dataset):
    def __init__(self, json_file, player_id, frame_rate=60, clip_duration=2, max_missing=5, shift_step=10):
        """
        json_file: Path to the single JSON file.
        player_id: Player ID for which data is being collected (e.g., 'player_1').
        frame_rate: Frames per second of the dataset (default is 60 FPS).
        clip_duration: Duration of each clip in seconds (e.g., 2 seconds).
        max_missing: Max allowed missing frames within a clip.
        shift_step: Step size for sliding the window (in number of frames).
        """
        self.json_file = json_file
        self.player_id = player_id
        self.num_frames = frame_rate * clip_duration  # 120 frames for a 2-second clip at 60 FPS
        self.max_missing = max_missing
        self.shift_step = shift_step
        self.actions = self.load_action_data()

    def load_action_data(self):
        """
        Load action data from the provided JSON file and extract valid frames for the specified player.
        """
        actions = []
        with open(self.json_file) as f:
            data = json.load(f)
            player_data = data['action_data']['players'].get(self.player_id, None)
            if player_data:
                frames = player_data['frames']
                # Filter frames with missing data (if any)
                valid_frames = [frame for frame in frames if frame is not None]
                if len(valid_frames) >= self.num_frames - self.max_missing:
                    actions.append(valid_frames)
        return actions

    def __len__(self):
        """
        Return the number of sliding windows we can extract from all actions.
        """
        total_windows = 0
        for frames in self.actions:
            total_windows += max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
        return total_windows

    def __getitem__(self, idx):
        """
        Get a specific time window (2-second clip) from the dataset.
        """
        current_idx = 0
        for frames in self.actions:
            num_windows = max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
            if idx < current_idx + num_windows:
                # Find the exact window within the action
                window_start = (idx - current_idx) * self.shift_step
                window_frames = frames[window_start:window_start + self.num_frames]
                # Convert the frames to a flat feature vector
                features = self.extract_features(window_frames)
                return torch.tensor(features, dtype=torch.float32)
            current_idx += num_windows
        raise IndexError("Index out of range")

    def extract_features(self, frames):
        """
        Flatten and combine the features for all frames in a window.
        """
        pose_features = []
        for frame in frames:
            # Assuming frame contains the structure you provided
            joint_angles_2d = [
                frame['joint_angles_2D']['right_leg']['ankle'],
                frame['joint_angles_2D']['right_leg']['knee'],
                frame['joint_angles_2D']['right_leg']['hip'],
                frame['joint_angles_2D']['left_leg']['ankle'],
                frame['joint_angles_2D']['left_leg']['knee'],
                frame['joint_angles_2D']['left_leg']['hip']
            ]
            angular_velocity_2d = [
                frame['angular_velocity_2D']['right_leg']['ankle'],
                frame['angular_velocity_2D']['right_leg']['knee'],
                frame['angular_velocity_2D']['right_leg']['hip'],
                frame['angular_velocity_2D']['left_leg']['ankle'],
                frame['angular_velocity_2D']['left_leg']['knee'],
                frame['angular_velocity_2D']['left_leg']['hip']
            ]
            linear_velocity_2d = np.linalg.norm(frame['linear_velocity_2D']['pelvis'])
            linear_acceleration_2d = np.linalg.norm(frame['linear_acceleration_2D']['pelvis'])
            joint_angles_3d = [
                frame['joint_angles_3D']['right_leg']['hip'],
                frame['joint_angles_3D']['right_leg']['knee'],
                frame['joint_angles_3D']['right_leg']['ankle'],
                frame['joint_angles_3D']['left_leg']['hip'],
                frame['joint_angles_3D']['left_leg']['knee'],
                frame['joint_angles_3D']['left_leg']['ankle']
            ]
            angular_velocity_3d = [
                frame['angular_velocity_3D']['right_leg']['hip'],
                frame['angular_velocity_3D']['right_leg']['knee'],
                frame['angular_velocity_3D']['right_leg']['ankle'],
                frame['angular_velocity_3D']['left_leg']['hip'],
                frame['angular_velocity_3D']['left_leg']['knee'],
                frame['angular_velocity_3D']['left_leg']['ankle']
            ]
            linear_velocity_3d = np.linalg.norm(frame['linear_velocity_3D']['pelvis'])
            linear_acceleration_3d = np.linalg.norm(frame['linear_acceleration_3D']['pelvis'])
            j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

            feature_vector = (
                joint_angles_2d +
                angular_velocity_2d +
                [linear_velocity_2d] + 
                [linear_acceleration_2d] +
                joint_angles_3d +
                angular_velocity_3d +
                [linear_velocity_3d] +
                [linear_acceleration_3d] +
                list(j3d_human)
            )
            pose_features.append(feature_vector)
        
        # Flatten the features for all frames into a single vector
        return np.concatenate(pose_features)

# Example usage with one JSON file
json_file = 'data_players/CHI_NYK/9/players_data.json'  # Replace with the path to a single JSON file
player_id = 'player_1'  # Assuming we want to extract data for player_1

dataset = PlayerDatasetSingleFile(json_file, player_id)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of iterating over the DataLoader
for batch in dataloader:
    print(batch.shape)  # Each batch will contain the flattened features of the player


torch.Size([32, 15960])
torch.Size([21, 15960])


In [11]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PlayerDatasetSingleFile(Dataset):
    def __init__(self, json_file, player_id, frame_rate=60, clip_duration=2, max_missing=5, shift_step=10):
        """
        json_file: Path to the single JSON file.
        player_id: Player ID for which data is being collected (e.g., 'player_1').
        frame_rate: Frames per second of the dataset (default is 60 FPS).
        clip_duration: Duration of each clip in seconds (e.g., 2 seconds).
        max_missing: Max allowed missing frames within a clip.
        shift_step: Step size for sliding the window (in number of frames).
        """
        self.json_file = json_file
        self.player_id = player_id
        self.num_frames = frame_rate * clip_duration  # 120 frames for a 2-second clip at 60 FPS
        self.max_missing = max_missing
        self.shift_step = shift_step
        self.actions = self.load_action_data()

    def load_action_data(self):
        """
        Load action data from the provided JSON file and extract valid frames for the specified player.
        """
        actions = []
        with open(self.json_file) as f:
            data = json.load(f)
            player_data = data['action_data']['players'].get(self.player_id, None)
            if player_data:
                frames = player_data['frames']
                # Filter frames with missing data (if any)
                valid_frames = [frame for frame in frames if frame is not None]
                if len(valid_frames) >= self.num_frames - self.max_missing:
                    actions.append(valid_frames)
        return actions

    def __len__(self):
        """
        Return the number of sliding windows we can extract from all actions.
        """
        total_windows = 0
        for frames in self.actions:
            total_windows += max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
        return total_windows

    def __getitem__(self, idx):
        """
        Get a specific time window (2-second clip) from the dataset.
        """
        current_idx = 0
        for frames in self.actions:
            num_windows = max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
            if idx < current_idx + num_windows:
                # Find the exact window within the action
                window_start = (idx - current_idx) * self.shift_step
                window_frames = frames[window_start:window_start + self.num_frames]

                # Get frame numbers for tracking
                frame_numbers = [frame['frame_number'] for frame in window_frames]

                # Convert the frames to a flat feature vector
                features = self.extract_features(window_frames)

                # Return features and frame numbers for tracking
                return torch.tensor(features, dtype=torch.float32), frame_numbers

            current_idx += num_windows
        raise IndexError("Index out of range")

    def extract_features(self, frames):
        """
        Flatten and combine the features for all frames in a window.
        """
        pose_features = []
        for frame in frames:
            # Extract and flatten features (as before)
            joint_angles_2d = [
                frame['joint_angles_2D']['right_leg']['ankle'],
                frame['joint_angles_2D']['right_leg']['knee'],
                frame['joint_angles_2D']['right_leg']['hip'],
                frame['joint_angles_2D']['left_leg']['ankle'],
                frame['joint_angles_2D']['left_leg']['knee'],
                frame['joint_angles_2D']['left_leg']['hip']
            ]
            angular_velocity_2d = [
                frame['angular_velocity_2D']['right_leg']['ankle'],
                frame['angular_velocity_2D']['right_leg']['knee'],
                frame['angular_velocity_2D']['right_leg']['hip'],
                frame['angular_velocity_2D']['left_leg']['ankle'],
                frame['angular_velocity_2D']['left_leg']['knee'],
                frame['angular_velocity_2D']['left_leg']['hip']
            ]
            linear_velocity_2d = np.linalg.norm(frame['linear_velocity_2D']['pelvis'])
            linear_acceleration_2d = np.linalg.norm(frame['linear_acceleration_2D']['pelvis'])
            joint_angles_3d = [
                frame['joint_angles_3D']['right_leg']['hip'],
                frame['joint_angles_3D']['right_leg']['knee'],
                frame['joint_angles_3D']['right_leg']['ankle'],
                frame['joint_angles_3D']['left_leg']['hip'],
                frame['joint_angles_3D']['left_leg']['knee'],
                frame['joint_angles_3D']['left_leg']['ankle']
            ]
            angular_velocity_3d = [
                frame['angular_velocity_3D']['right_leg']['hip'],
                frame['angular_velocity_3D']['right_leg']['knee'],
                frame['angular_velocity_3D']['right_leg']['ankle'],
                frame['angular_velocity_3D']['left_leg']['hip'],
                frame['angular_velocity_3D']['left_leg']['knee'],
                frame['angular_velocity_3D']['left_leg']['ankle']
            ]
            linear_velocity_3d = np.linalg.norm(frame['linear_velocity_3D']['pelvis'])
            linear_acceleration_3d = np.linalg.norm(frame['linear_acceleration_3D']['pelvis'])
            j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

            feature_vector = (
                joint_angles_2d +
                angular_velocity_2d +
                [linear_velocity_2d] + 
                [linear_acceleration_2d] +
                joint_angles_3d +
                angular_velocity_3d +
                [linear_velocity_3d] +
                [linear_acceleration_3d] +
                list(j3d_human)
            )
            pose_features.append(feature_vector)
        
        # Flatten the features for all frames into a single vector
        return np.concatenate(pose_features)

# Example usage with one JSON file
json_file = 'data_players/CHI_NYK/9/players_data.json'  # Replace with the path to a single JSON file
player_id = 'player_1'  # Assuming we want to extract data for player_1

dataset = PlayerDatasetSingleFile(json_file, player_id)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of iterating over the DataLoader and printing frame ranges
for features, frame_numbers in dataloader:
    print(f"Features shape: {features.shape}")
    #print(f"Frame numbers in this window: {frame_numbers}")


Features shape: torch.Size([32, 15960])
Features shape: torch.Size([21, 15960])


In [12]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PlayerDatasetMultiFile(Dataset):
    def __init__(self, json_file, player_ids, frame_rate=60, clip_duration=2, max_missing=5, shift_step=10):
        """
        json_file: Path to the single JSON file.
        player_ids: List of player IDs (e.g., ['player_1', 'player_2']).
        frame_rate: Frames per second of the dataset (default is 60 FPS).
        clip_duration: Duration of each clip in seconds (e.g., 2 seconds).
        max_missing: Max allowed missing frames within a clip.
        shift_step: Step size for sliding the window (in number of frames).
        """
        self.json_file = json_file
        self.player_ids = player_ids  # A list of player IDs
        self.num_frames = frame_rate * clip_duration  # 120 frames for a 2-second clip at 60 FPS
        self.max_missing = max_missing
        self.shift_step = shift_step
        self.actions = self.load_action_data()

    def load_action_data(self):
        """
        Load action data from the provided JSON file and extract valid frames for the specified players.
        """
        actions = {player_id: [] for player_id in self.player_ids}  # Store actions for each player
        with open(self.json_file) as f:
            data = json.load(f)
            for player_id in self.player_ids:
                player_data = data['action_data']['players'].get(player_id, None)
                if player_data:
                    frames = player_data['frames']
                    # Filter frames with missing data (if any)
                    valid_frames = [frame for frame in frames if frame is not None]
                    if len(valid_frames) >= self.num_frames - self.max_missing:
                        actions[player_id].append(valid_frames)
        return actions

    def __len__(self):
        """
        Return the number of sliding windows we can extract for the players.
        """
        total_windows = 0
        for player_id, frames_list in self.actions.items():
            for frames in frames_list:
                total_windows += max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
        return total_windows

    def __getitem__(self, idx):
        """
        Get a specific time window (2-second clip) from the dataset for all players.
        """
        current_idx = 0
        for player_id, frames_list in self.actions.items():
            for frames in frames_list:
                num_windows = max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
                if idx < current_idx + num_windows:
                    # Find the exact window within the action
                    window_start = (idx - current_idx) * self.shift_step
                    window_frames = frames[window_start:window_start + self.num_frames]

                    # Get frame numbers for tracking
                    frame_numbers = [frame['frame_number'] for frame in window_frames]

                    # Convert the frames to a flat feature vector
                    features = self.extract_features(window_frames)

                    # Return features and frame numbers for tracking, including player_id
                    return torch.tensor(features, dtype=torch.float32), frame_numbers, player_id

                current_idx += num_windows
        raise IndexError("Index out of range")

    def extract_features(self, frames):
        """
        Flatten and combine the features for all frames in a window.
        """
        pose_features = []
        for frame in frames:
            # Extract and flatten features (same as before)
            joint_angles_2d = [
                frame['joint_angles_2D']['right_leg']['ankle'],
                frame['joint_angles_2D']['right_leg']['knee'],
                frame['joint_angles_2D']['right_leg']['hip'],
                frame['joint_angles_2D']['left_leg']['ankle'],
                frame['joint_angles_2D']['left_leg']['knee'],
                frame['joint_angles_2D']['left_leg']['hip']
            ]
            angular_velocity_2d = [
                frame['angular_velocity_2D']['right_leg']['ankle'],
                frame['angular_velocity_2D']['right_leg']['knee'],
                frame['angular_velocity_2D']['right_leg']['hip'],
                frame['angular_velocity_2D']['left_leg']['ankle'],
                frame['angular_velocity_2D']['left_leg']['knee'],
                frame['angular_velocity_2D']['left_leg']['hip']
            ]
            linear_velocity_2d = np.linalg.norm(frame['linear_velocity_2D']['pelvis'])
            linear_acceleration_2d = np.linalg.norm(frame['linear_acceleration_2D']['pelvis'])
            joint_angles_3d = [
                frame['joint_angles_3D']['right_leg']['hip'],
                frame['joint_angles_3D']['right_leg']['knee'],
                frame['joint_angles_3D']['right_leg']['ankle'],
                frame['joint_angles_3D']['left_leg']['hip'],
                frame['joint_angles_3D']['left_leg']['knee'],
                frame['joint_angles_3D']['left_leg']['ankle']
            ]
            angular_velocity_3d = [
                frame['angular_velocity_3D']['right_leg']['hip'],
                frame['angular_velocity_3D']['right_leg']['knee'],
                frame['angular_velocity_3D']['right_leg']['ankle'],
                frame['angular_velocity_3D']['left_leg']['hip'],
                frame['angular_velocity_3D']['left_leg']['knee'],
                frame['angular_velocity_3D']['left_leg']['ankle']
            ]
            linear_velocity_3d = np.linalg.norm(frame['linear_velocity_3D']['pelvis'])
            linear_acceleration_3d = np.linalg.norm(frame['linear_acceleration_3D']['pelvis'])
            j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

            feature_vector = (
                joint_angles_2d +
                angular_velocity_2d +
                [linear_velocity_2d] + 
                [linear_acceleration_2d] +
                joint_angles_3d +
                angular_velocity_3d +
                [linear_velocity_3d] +
                [linear_acceleration_3d] +
                list(j3d_human)
            )
            pose_features.append(feature_vector)
        
        # Flatten the features for all frames into a single vector
        return np.concatenate(pose_features)

# Example usage with one JSON file and multiple player IDs
json_file = 'data_players/CHI_NYK/9/players_data.json' 
player_ids = ['player_1', 'player_2'] 

dataset = PlayerDatasetMultiFile(json_file, player_ids)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of iterating over the DataLoader and printing frame ranges for multiple players
for features, frame_numbers, player_id in dataloader:
    print(f"Features shape: {features.shape}")
    #print(f"Frame numbers in this window for {player_id}: {frame_numbers}")


Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([8, 15960])


In [13]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class PlayerDatasetAllPlayers(Dataset):
    def __init__(self, json_file, frame_rate=60, clip_duration=2, max_missing=5, shift_step=10):
        """
        json_file: Path to the single JSON file.
        frame_rate: Frames per second of the dataset (default is 60 FPS).
        clip_duration: Duration of each clip in seconds (e.g., 2 seconds).
        max_missing: Max allowed missing frames within a clip.
        shift_step: Step size for sliding the window (in number of frames).
        """
        self.json_file = json_file
        self.num_frames = frame_rate * clip_duration  # 120 frames for a 2-second clip at 60 FPS
        self.max_missing = max_missing
        self.shift_step = shift_step
        self.actions = self.load_action_data()

    def load_action_data(self):
        """
        Load action data from the provided JSON file and extract valid frames for all players.
        """
        actions = {}  # Dictionary to store frames for each player
        with open(self.json_file) as f:
            data = json.load(f)
            players = data['action_data']['players']
            for player_id, player_data in players.items():
                frames = player_data['frames']
                # Filter frames with missing data (if any)
                valid_frames = [frame for frame in frames if frame is not None]
                if len(valid_frames) >= self.num_frames - self.max_missing:
                    actions[player_id] = valid_frames
        return actions

    def __len__(self):
        """
        Return the number of sliding windows we can extract for all players.
        """
        total_windows = 0
        for player_id, frames in self.actions.items():
            total_windows += max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
        return total_windows

    def __getitem__(self, idx):
        """
        Get a specific time window (2-second clip) from the dataset for all players.
        """
        current_idx = 0
        for player_id, frames in self.actions.items():
            num_windows = max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
            if idx < current_idx + num_windows:
                # Find the exact window within the action
                window_start = (idx - current_idx) * self.shift_step
                window_frames = frames[window_start:window_start + self.num_frames]

                # Get frame numbers for tracking
                frame_numbers = [frame['frame_number'] for frame in window_frames]

                # Convert the frames to a flat feature vector
                features = self.extract_features(window_frames)

                # Return features and frame numbers for tracking, including player_id
                return torch.tensor(features, dtype=torch.float32), frame_numbers, player_id

            current_idx += num_windows
        raise IndexError("Index out of range")

    def extract_features(self, frames):
        """
        Flatten and combine the features for all frames in a window.
        """
        pose_features = []
        for frame in frames:
            # Extract and flatten features (same as before)
            joint_angles_2d = [
                frame['joint_angles_2D']['right_leg']['ankle'],
                frame['joint_angles_2D']['right_leg']['knee'],
                frame['joint_angles_2D']['right_leg']['hip'],
                frame['joint_angles_2D']['left_leg']['ankle'],
                frame['joint_angles_2D']['left_leg']['knee'],
                frame['joint_angles_2D']['left_leg']['hip']
            ]
            angular_velocity_2d = [
                frame['angular_velocity_2D']['right_leg']['ankle'],
                frame['angular_velocity_2D']['right_leg']['knee'],
                frame['angular_velocity_2D']['right_leg']['hip'],
                frame['angular_velocity_2D']['left_leg']['ankle'],
                frame['angular_velocity_2D']['left_leg']['knee'],
                frame['angular_velocity_2D']['left_leg']['hip']
            ]
            linear_velocity_2d = np.linalg.norm(frame['linear_velocity_2D']['pelvis'])
            linear_acceleration_2d = np.linalg.norm(frame['linear_acceleration_2D']['pelvis'])
            joint_angles_3d = [
                frame['joint_angles_3D']['right_leg']['hip'],
                frame['joint_angles_3D']['right_leg']['knee'],
                frame['joint_angles_3D']['right_leg']['ankle'],
                frame['joint_angles_3D']['left_leg']['hip'],
                frame['joint_angles_3D']['left_leg']['knee'],
                frame['joint_angles_3D']['left_leg']['ankle']
            ]
            angular_velocity_3d = [
                frame['angular_velocity_3D']['right_leg']['hip'],
                frame['angular_velocity_3D']['right_leg']['knee'],
                frame['angular_velocity_3D']['right_leg']['ankle'],
                frame['angular_velocity_3D']['left_leg']['hip'],
                frame['angular_velocity_3D']['left_leg']['knee'],
                frame['angular_velocity_3D']['left_leg']['ankle']
            ]
            linear_velocity_3d = np.linalg.norm(frame['linear_velocity_3D']['pelvis'])
            linear_acceleration_3d = np.linalg.norm(frame['linear_acceleration_3D']['pelvis'])
            j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

            feature_vector = (
                joint_angles_2d +
                angular_velocity_2d +
                [linear_velocity_2d] + 
                [linear_acceleration_2d] +
                joint_angles_3d +
                angular_velocity_3d +
                [linear_velocity_3d] +
                [linear_acceleration_3d] +
                list(j3d_human)
            )
            pose_features.append(feature_vector)
        
        # Flatten the features for all frames into a single vector
        return np.concatenate(pose_features)

# Example usage with one JSON file and all players
json_file = 'data_players/CHI_NYK/9/players_data.json' 

dataset = PlayerDatasetAllPlayers(json_file)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of iterating over the DataLoader and printing frame ranges for all players
for features, frame_numbers, player_id in dataloader:
    print(f"Features shape: {features.shape}")
    #print(f"Frame numbers in this window for {player_id}: {frame_numbers}")


Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([32, 15960])
Features shape: torch.Size([29, 15960])


In [4]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

test_features_nan= []

class PlayerDatasetAllPlayers(Dataset):
    def __init__(self, json_file, frame_rate=60, clip_duration=2, max_missing=5, shift_step=10):
        """
        json_file: Path to the single JSON file.
        frame_rate: Frames per second of the dataset (default is 60 FPS).
        clip_duration: Duration of each clip in seconds (e.g., 2 seconds).
        max_missing: Max allowed missing frames within a clip.
        shift_step: Step size for sliding the window (in number of frames).
        """
        self.json_file = json_file
        self.num_frames = frame_rate * clip_duration  # 120 frames for a 2-second clip at 60 FPS
        self.max_missing = max_missing
        self.shift_step = shift_step
        self.actions = self.load_action_data()

    def load_action_data(self):
        """
        Load action data from the provided JSON file and extract valid frames for all players.
        """
        actions = {}  # Dictionary to store frames for each player
        with open(self.json_file) as f:
            data = json.load(f)
            players = data['action_data']['players']
            for player_id, player_data in players.items():
                frames = player_data['frames']
                # Filter frames with missing data (if any)
                valid_frames = [frame for frame in frames if frame is not None]
                if len(valid_frames) >= self.num_frames - self.max_missing:
                    actions[player_id] = valid_frames
        return actions

    def __len__(self):
        """
        Return the number of sliding windows we can extract for all players.
        """
        total_windows = 0
        for player_id, frames in self.actions.items():
            total_windows += max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
        return total_windows

    def __getitem__(self, idx):
        """
        Get a specific time window (2-second clip) from the dataset for all players.
        """
        current_idx = 0
        for player_id, frames in self.actions.items():
            num_windows = max(0, (len(frames) - self.num_frames) // self.shift_step + 1)
            if idx < current_idx + num_windows:
                # Find the exact window within the action
                window_start = (idx - current_idx) * self.shift_step
                window_frames = frames[window_start:window_start + self.num_frames]

                # Get frame numbers for tracking
                frame_numbers = [frame['frame_number'] for frame in window_frames]

                # Convert the frames to a flat feature vector
                features = self.extract_features(window_frames, player_id, frame_numbers)

                # Return features and frame numbers for tracking, including player_id
                return torch.tensor(features, dtype=torch.float32), frame_numbers, player_id

            current_idx += num_windows
        raise IndexError("Index out of range")

    def extract_features(self, frames, player_id, frame_numbers):
        """
        Flatten and combine the features for all frames in a window.
        Also checks for NaN/Inf values in individual features before concatenating.
        """
        pose_features = []
        for i, frame in enumerate(frames):
            # Extract and flatten features (same as before)
            joint_angles_2d = [
                frame['joint_angles_2D']['right_leg']['ankle'],
                frame['joint_angles_2D']['right_leg']['knee'],
                frame['joint_angles_2D']['right_leg']['hip'],
                frame['joint_angles_2D']['left_leg']['ankle'],
                frame['joint_angles_2D']['left_leg']['knee'],
                frame['joint_angles_2D']['left_leg']['hip']
            ]
            angular_velocity_2d = [
                frame['angular_velocity_2D']['right_leg']['ankle'],
                frame['angular_velocity_2D']['right_leg']['knee'],
                frame['angular_velocity_2D']['right_leg']['hip'],
                frame['angular_velocity_2D']['left_leg']['ankle'],
                frame['angular_velocity_2D']['left_leg']['knee'],
                frame['angular_velocity_2D']['left_leg']['hip']
            ]
            joint_angles_3d = [
                frame['joint_angles_3D']['right_leg']['hip'],
                frame['joint_angles_3D']['right_leg']['knee'],
                frame['joint_angles_3D']['right_leg']['ankle'],
                frame['joint_angles_3D']['left_leg']['hip'],
                frame['joint_angles_3D']['left_leg']['knee'],
                frame['joint_angles_3D']['left_leg']['ankle']
            ]
            angular_velocity_3d = [
                frame['angular_velocity_3D']['right_leg']['hip'],
                frame['angular_velocity_3D']['right_leg']['knee'],
                frame['angular_velocity_3D']['right_leg']['ankle'],
                frame['angular_velocity_3D']['left_leg']['hip'],
                frame['angular_velocity_3D']['left_leg']['knee'],
                frame['angular_velocity_3D']['left_leg']['ankle']
            ]
            lin_vel_3d_knee_right = np.linalg.norm(frame['linear_velocity_3D']['right_leg']['knee'])
            lin_acc_3d_knee_right = np.linalg.norm(frame['linear_acceleration_3D']['right_leg']['knee'])
            lin_vel_3d_ankle_right = np.linalg.norm(frame['linear_velocity_3D']['right_leg']['ankle'])
            lin_acc_3d_ankle_right = np.linalg.norm(frame['linear_acceleration_3D']['right_leg']['ankle'])

            lin_vel_3d_knee_left = np.linalg.norm(frame['linear_velocity_3D']['left_leg']['knee'])
            lin_acc_3d_knee_left = np.linalg.norm(frame['linear_acceleration_3D']['left_leg']['knee'])
            lin_vel_3d_ankle_left = np.linalg.norm(frame['linear_velocity_3D']['left_leg']['ankle'])
            lin_acc_3d_ankle_left = np.linalg.norm(frame['linear_acceleration_3D']['left_leg']['ankle'])


            
            j3d_human = np.array(frame['j3d_human']['coordinates']).flatten()

            # Check for NaNs or Infs in individual features
            self.check_nan_inf(np.array(joint_angles_2d), player_id, frame['frame_number'], "joint_angles_2d")
            self.check_nan_inf(np.array(angular_velocity_2d), player_id, frame['frame_number'], "angular_velocity_2d")
            self.check_nan_inf(np.array(joint_angles_3d), player_id, frame['frame_number'], "joint_angles_3d")
            self.check_nan_inf(np.array(angular_velocity_3d), player_id, frame['frame_number'], "angular_velocity_3d")
            #self.check_nan_inf(np.array([linear_velocity_3d]), player_id, frame['frame_number'], "linear_velocity_3d")
            #self.check_nan_inf(np.array([linear_acceleration_3d]), player_id, frame['frame_number'], "linear_acceleration_3d")
            self.check_nan_inf(j3d_human, player_id, frame['frame_number'], "j3d_human")

            # Combine the features
            feature_vector = (
                joint_angles_2d +
                angular_velocity_2d +
                joint_angles_3d +
                angular_velocity_3d +
                [lin_vel_3d_knee_right] +
                [lin_acc_3d_knee_right] +
                [lin_vel_3d_ankle_right] +
                [lin_acc_3d_ankle_right] +
                [lin_vel_3d_knee_left] +
                [lin_acc_3d_knee_left] +
                [lin_vel_3d_ankle_left] +
                [lin_acc_3d_ankle_left] +
                list(j3d_human)
            )
            pose_features.append(feature_vector)

        # Flatten the features for all frames into a single vector
        flattened_features = np.concatenate(pose_features)

        return flattened_features


    def check_nan_inf(self, features, player_id, frame_numbers, feature_name):
        """
        Check for NaN or Inf values in features and report the player, frame, and problematic feature.
        """

        if np.isnan(features).any():
            nan_idx = np.where(np.isnan(features))[0]
            print(f"NaN detected for player {player_id} in frames {frame_numbers} for feature {feature_name}. NaN in features at index: {nan_idx}")
            if feature_name not in test_features_nan:
                test_features_nan.append(feature_name)
        if np.isinf(features).any():
            inf_idx = np.where(np.isinf(features))[0]
            print(f"Inf detected for player {player_id} in frames {frame_numbers} for feature {feature_name}. Inf in features at index: {inf_idx}")

# Example usage with one JSON file and all players
json_file = 'data_players/CHI_NYK/451/players_data.json'

dataset = PlayerDatasetAllPlayers(json_file)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of iterating over the DataLoader and printing frame ranges for all players
for features, frame_numbers, player_id in dataloader:
    print(f"Features shape: {features.shape}")
    # The NaN or Inf check will happen inside the Dataset class


KeyError: 'right_leg'

In [5]:
import json
with open('data_players/CHI_NYK/9/players_data.json') as f:
    data_9 = json.load(f)

for i in data_9['action_data']['players']['player_7']['frames'][370:376]:
    #print all thing related to angular features
    print('#####')
    print(i['joint_angles_2D']['right_leg'])
    print(i['angular_velocity_2D']['right_leg'])
    print(i['angular_acceleration_2D']['right_leg'])
    print(i['joint_angles_3D']['right_leg'])
    print(i['angular_velocity_3D']['right_leg'])
    print(i['angular_acceleration_3D']['right_leg'])
    print(i['linear_velocity_3D']['right_leg'])
    print('#####')

    

#####
{'ankle': 3.02, 'knee': 3.05, 'hip': 1.88}
{'ankle': 0.36, 'knee': -0.6, 'hip': -3.24}
{'ankle': -15.84, 'knee': -23.04, 'hip': 5.76}
{'ankle': 1.88, 'knee': 2.16, 'hip': 2.02}
{'ankle': 2.04, 'knee': 2.04, 'hip': -0.6}
{'ankle': 21.6, 'knee': 53.28, 'hip': 20.16}
{'ankle': [0.72, 0.48, -0.48], 'knee': [0.0, -0.12, -0.36]}
#####
#####
{'ankle': 2.82, 'knee': 3.05, 'hip': 1.88}
{'ankle': -2.52, 'knee': 0.24, 'hip': -3.48}
{'ankle': -36.0, 'knee': -4.32, 'hip': -34.56}
{'ankle': 1.87, 'knee': 2.17, 'hip': 2.04}
{'ankle': 1.56, 'knee': 2.28, 'hip': -0.24}
{'ankle': 14.4, 'knee': 59.04, 'hip': 8.64}
{'ankle': [0.36, 0.36, -0.24], 'knee': [-0.36, -0.24, -0.24]}
#####
#####
{'ankle': 2.82, 'knee': 3.05, 'hip': 1.88}
{'ankle': -3.48, 'knee': 0.24, 'hip': -3.48}
{'ankle': -87.84, 'knee': -12.96, 'hip': -34.56}
{'ankle': 1.91, 'knee': 2.19, 'hip': 1.98}
{'ankle': 1.56, 'knee': 1.44, 'hip': -2.04}
{'ankle': 17.28, 'knee': 47.52, 'hip': -21.6}
{'ankle': [0.0, 1.2, -0.12], 'knee': [-0.84, 0.

In [16]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset

def load_all_json_files(root_folder):
    """
    Load all player_data.json files from all subdirectories under the root folder.
    """
    json_files = []
    print(f"Loading JSON files from: {root_folder}")
    for subdir in os.listdir(root_folder):
        folder_path = os.path.join(root_folder, subdir)
        json_file_path = os.path.join(folder_path, 'players_data.json')
        if os.path.exists(json_file_path):
            json_files.append(json_file_path)
    return json_files

def create_dataloaders_for_all_folders(root_folder, batch_size=32, shuffle=True):
    """
    Create dataloaders for all JSON files inside subdirectories in the root folder.
    """
    json_files = load_all_json_files(root_folder)
    print(f"Found {len(json_files)} JSON files.")
    datasets = []
    for json_file in json_files:
        print(f"Creating dataset for: {json_file}")
        dataset = PlayerDatasetAllPlayers(json_file)
        datasets.append(dataset)
    
    # Option 1: Combine all datasets into one using ConcatDataset
    combined_dataset = ConcatDataset(datasets)
    dataloader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=shuffle)
    
    return dataloader

# Specify the root folder
root_folder = 'data_players/CHI_NYK'

# Create the dataloader for all JSON files inside the subdirectories
dataloader = create_dataloaders_for_all_folders(root_folder)

# Example of iterating over the combined DataLoader and printing frame ranges for all players
for features, frame_numbers, player_id in dataloader:
    print(f"Features shape: {features.shape}")
    # The NaN or Inf check will happen inside the Dataset class


Loading JSON files from: data_players/CHI_NYK
Found 399 JSON files.
Creating dataset for: data_players/CHI_NYK\101\players_data.json
Creating dataset for: data_players/CHI_NYK\102\players_data.json
Creating dataset for: data_players/CHI_NYK\103\players_data.json
Creating dataset for: data_players/CHI_NYK\104\players_data.json
Creating dataset for: data_players/CHI_NYK\105\players_data.json
Creating dataset for: data_players/CHI_NYK\107\players_data.json
Creating dataset for: data_players/CHI_NYK\109\players_data.json
Creating dataset for: data_players/CHI_NYK\110\players_data.json
Creating dataset for: data_players/CHI_NYK\113\players_data.json
Creating dataset for: data_players/CHI_NYK\114\players_data.json
Creating dataset for: data_players/CHI_NYK\115\players_data.json
Creating dataset for: data_players/CHI_NYK\116\players_data.json
Creating dataset for: data_players/CHI_NYK\118\players_data.json
Creating dataset for: data_players/CHI_NYK\12\players_data.json
Creating dataset for: d

In [81]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define AutoEncoder Architecture for Flat Features
class PoseAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(PoseAutoencoder, self).__init__()
        
        # Encoder for flattened features
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )
        
        # Decoder (inverse of encoder)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # Use Sigmoid for normalized outputs between 0 and 1
        )
    
    def forward(self, x):
        # Pass through encoder and decoder
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# Assuming the DataLoader from previous example is available
# Initialize model, loss function, and optimizer
input_dim = 15960  # Replace with the actual flattened feature size
latent_dim = 64  # You can experiment with different latent dimensions

model = PoseAutoencoder(input_dim=input_dim, latent_dim=latent_dim)
criterion = nn.MSELoss()  # Mean Squared Error for reconstruction
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop using DataLoader
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (features, frame_numbers, player_id) in enumerate(dataloader):
        # The features are the flattened features we will train on
        batch = features  # features is already a tensor from DataLoader
        
        print(torch.isnan(features).sum(), "nan")  # To check if NaN values exist in the input
        print(torch.isinf(features).sum(), 'inf')  # 

        # Zero the gradient
        optimizer.zero_grad()
        
        # Forward pass: Encode and Decode
        reconstructed, latent = model(batch)
        
        # Calculate the loss (reconstruction loss)
        loss = criterion(reconstructed, batch)
        
        # Backpropagation
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Accumulate the loss
        running_loss += loss.item()
    
    # Print average loss for the epoch
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}')

# After training, the latent representation (embedding) can be used for clustering, retrieval, etc.



tensor(1487) nan
tensor(0) inf
tensor(1132) nan
tensor(0) inf
tensor(1038) nan
tensor(0) inf
tensor(1393) nan
tensor(0) inf
tensor(1184) nan
tensor(0) inf
tensor(1474) nan
tensor(0) inf
tensor(1332) nan
tensor(0) inf


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define AutoEncoder Architecture
class PoseAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(PoseAutoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # Match the input data range
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# Load your dataset (you'll need to transform it into a proper input matrix)
# Assuming X_train is your input matrix with pose features
# X_train = your_preprocessed_dataset

input_dim = X_train.shape[1]  # Number of features (2D and 3D joint angles, velocities, etc.)
latent_dim = 64  # You can experiment with different latent dimensions

# Initialize model, loss function and optimizer
model = PoseAutoencoder(input_dim=input_dim, latent_dim=latent_dim)
criterion = nn.MSELoss()  # Mean Squared Error for reconstruction
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 100
batch_size = 32

for epoch in range(num_epochs):
    model.train()
    for i in range(0, len(X_train), batch_size):
        batch = X_train[i:i+batch_size]
        batch = torch.tensor(batch, dtype=torch.float32)
        
        optimizer.zero_grad()
        reconstructed, latent = model(batch)
        loss = criterion(reconstructed, batch)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Once trained, the latent representation (embedding) can be used for clustering
